In [60]:
import os 
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import openpyxl

'Lasso is linear regression with L1 penalty on coefficients (shrinks some coefficients to exactly zero).',
'Reduce overfitting and perform feature selection by shrinking coefficients.',
'Minimize sum of squared errors + alpha * sum(|beta_j|).',
'Higher alpha → stronger shrinkage → more coefficients driven to zero.',
'L1 penalty leads to sparse coefficients (some become exactly 0).',
'Features must be scaled because penalty depends on coefficient magnitude which depends on feature scale.',
'Use coefficients on original scale (we computed them) to interpret effect on target (mpg).'

In [61]:
df=sns.load_dataset("mpg")

In [62]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    str    
 8   name          398 non-null    str    
dtypes: float64(4), int64(3), str(2)
memory usage: 35.9 KB


In [63]:



df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


In [64]:
df.isnull().sum()

mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64

In [65]:
# Encode 'origin' because it has text like "usa", "japan", "europe"
df['origin']=df['origin'].map({'usa':1,'europe':2,'japan':3})

# %%
#  Quick look & keep required columns (choose useful numeric features)
# We'll keep: mpg (target), cylinders, displacement, horsepower, weight, acceleration, model_year, origin
df = df[['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin', 'name']].copy()


In [66]:
# Handle missing values (no inplace)
# horsepower has missing values in seaborn mpg -> fill with median
df['horsepower'] = df['horsepower'].fillna(df['horsepower'].median())


In [67]:
# 8. Define X and y
# target: mpg (continuous)
X = df[['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']].copy()
y = df['mpg'].copy()

# %%
# 9. Train-test split (one line)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
#print(X_train.head())
print("X test",X_test[:5])
print("Y test\n",y_test[:5])
#print(y_train[:5])

X test      cylinders  displacement  horsepower  weight  acceleration  model_year  \
198          4          91.0        53.0    1795          17.4          76   
396          4         120.0        79.0    2625          18.6          82   
33           6         232.0       100.0    2634          13.0          71   
208          8         318.0       150.0    3940          13.2          76   
93           8         318.0       150.0    4237          14.5          73   

     origin  
198       3  
396       1  
33        1  
208       1  
93        1  
Y test
 198    33.0
396    28.0
33     19.0
208    13.0
93     14.0
Name: mpg, dtype: float64


In [68]:
#Scale features (important for Lasso)
scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)

# Save scaler params for coefficient unscaling
scales = scaler.scale_
means = scaler.mean_

In [69]:
# 11. Train Lasso model (choose alpha; smaller alpha -> less shrinkage)
alpha = 0.1   # change to 1.0 or 0.01 to demonstrate effect
lasso = Lasso(alpha=alpha,random_state=42,max_iter=10000)
lasso.fit(X_train_s,y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.1
,"max_iter max_iter: int, default=1000The maximum number of iterations.",10000
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'
Name,Type,Value


In [70]:
# Coefficients on scaled features
coef_scaled=lasso.coef_.copy()
intercept_scaled=lasso.intercept_   # corresponds to features in feature_names

# Convert coefficients to original feature scale so they're easy to interpret:
# If model was trained on Xs = (X - mean)/scale and y not scaled:
# y = intercept_s + Xs @ coef_s
# Replace Xs = (X - mean)/scale => y = intercept_s + ((X - mean)/scale) @ coef_s
# => y = intercept_s - sum((mean/scale)*coef_s) + X @ (coef_s / scale)
coef_orig = coef_scaled / scales
intercept_orig = intercept_scaled - np.sum((means / scales) * coef_scaled)


In [71]:
#predictions
y_train_predict=lasso.predict(X_train_s)
y_test_predict=lasso.predict(X_test_s)
print(y_train_predict[:5]) # printing numpy ,if u want print in pandas use head() 
print(y_test_predict[:5])

[13.26414284 24.13916677 20.52695268 23.41068107 20.09603852]
[32.44844665 29.59716027 20.78199695 16.85704032 12.93647848]


In [72]:
#residuals
resid_train= y_train.values  - y_train_predict
resid_test = y_test.values  - y_test_predict
print(resid_train[:5])
print(resid_test[:5])

[ 1.73585716 -3.13916677  0.47304732 -2.91068107  0.90396148]
[ 0.55155335 -1.59716027 -1.78199695 -3.85704032  1.06352152]


In [73]:
# Evaluation metrics (test)
mse=mean_squared_error(y_test,y_test_predict)
rmse=np.sqrt(mse)
mae=mean_absolute_error(y_test,y_test_predict)
r2=r2_score(y_test,y_test_predict)
print(mse)
print(rmse)
print(mae)
print(r2)

9.29981293366506
3.049559465507282
2.362382901943143
0.8379579730554979


In [74]:
#print in a book like,
#RawData
#Training Data
#Testing Data
#Model Summary (Testing-Metrics)
#coefficents+intercept
#Training Predictions (include Actual_mpg	Pred_mpg	Residual)
#Testing Predictions
# Residual Summary (optional) Metrics for Test & train data


In [77]:
os.chdir(r"F:\DataScience")

with pd.ExcelWriter("Filename11.xlsx",engine='openpyxl') as writer:
    df.to_excel(writer,sheet_name="RawData")
    X_train.to_excel(writer,sheet_name="TrainingData")
    X_test.to_excel(writer,sheet_name="TestingData")

    metrics=pd.DataFrame({'Metric':['RMSE','MAE','R2'],'Value':[rmse,mae,r2]}) #with heading
    # or metrics=pd.DataFrame({'RMSE':[rmse],'MAE':[mae],'R2':[r2]})
    metrics.to_excel(writer,sheet_name="Metrics")

   
    #[intercept_scaled] use this [] for single values for the below statement
    #here required ignore_index=True,because intercept row taking index 0,andcoeff also 0 
    coeff=pd.DataFrame({'Features':X_train.columns,'Coef_scaled':coef_scaled,'Coef_Original':coef_orig})
    intercept=pd.DataFrame({'Features':'intercept','Coef_scaled':[intercept_scaled],'Coef_Original':[intercept_orig]})
    coeff_inter=pd.concat([intercept,coeff],ignore_index=True)
    coeff_inter.to_excel(writer,sheet_name="Coefficients")

   
    #Training Predictions (SRN)include Actual_mpg	Pred_mpg	Residual)
     #Add Serial Number for traceability
    #df.insert(0,'SR',range(1,len(df)+1))
    train_df = df.loc[X_train.index].reset_index(drop=True)
    train_result = train_df.copy()
    train_result['Actual_mpg'] = y_train.values
    train_result['Pred_mpg'] = np.round(y_train_predict,3)
    train_result['Residual'] = np.round(resid_train,3)
    train_result.to_excel(writer,sheet_name="Training Predictions",index=False)

    test_df = df.loc[X_test.index].reset_index(drop=True)
    test_result = test_df.copy()
    test_result['Actual_mpg'] = y_test.values
    test_result['Pred_mpg'] = np.round(y_test_predict, 3)
    test_result['Residual'] = np.round(resid_test, 3)
    test_result.to_excel(writer,sheet_name="Testing Predictions",index=False)

    resid_summary = pd.DataFrame({
    'Dataset': ['Train', 'Test'],
    'Count': [len(resid_train), len(resid_test)],
    'RMSE': [np.sqrt(mean_squared_error(y_train, y_train_predict)), rmse],
    'MAE': [mean_absolute_error(y_train, y_train_predict), mae],
    'R2': [r2_score(y_train, y_train_predict), r2]
    })
    resid_summary.to_excel(writer, sheet_name='Residual Summary', index=False)

    

In [86]:
# Optional: plot residuals (save figure)
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.scatter(test_result['Pred_mpg'], test_result['Residual'], alpha=0.7)
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Predicted MPG')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Lasso Residuals (Test)')
plt.tight_layout()
plt.savefig('Lasso_Residuals_Test.png')
plt.close()